# Model F: Optimized Score Fusion (85% BM25 + 15% Transformer)

This notebook implements the best-performing model from the full comparative analysis: a **weighted score fusion** between BM25 and Transformer embeddings.

## Key differences from previous notebooks

### 1. Type-Aware Level 2 Evaluation
Previous notebooks used a **generic** subcategory extractor that pooled all metadata columns together. This creates spurious cross-type matches (e.g., a hotel's price range being compared against a restaurant's cuisine type).

This notebook uses a **type-aware** extractor that applies only the relevant attributes per `typeR`:
- `H` (Hotel) → `priceRange` only
- `R` (Restaurant) → `restaurantType` + `restaurantTypeCuisine` only
- `A` / `AP` (Attraction) → `activiteSubType` only

### 2. Corpus-Wide TF-IDF for Keyword Extraction
Previous notebooks fit TF-IDF **per place** (word frequency relative to that place's own reviews). This notebook fits TF-IDF **on the whole corpus** (`max_features=5000`), making word weights relative to the entire dataset — a more standard and discriminative approach.

### 3. Hybrid Score Fusion
BM25 is excellent at exact keyword matching (Level 1), while Transformer embeddings add semantic depth (Level 2). Fusing them with **0.85 BM25 + 0.15 Transformer** achieves the best of both worlds.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import time
import warnings
warnings.filterwarnings('ignore')

### 1. Load Data & Train/Test Split (Same Seed)

In [ ]:
df_reviews = pd.read_csv('prepared_reviews.csv')
df_places  = pd.read_csv('Tripadvisor.csv', low_memory=False)

eval_cols = ['id', 'typeR', 'activiteSubType', 'restaurantType', 'restaurantTypeCuisine', 'priceRange']
df_places = df_places[eval_cols].copy()

df_merged = pd.merge(df_reviews, df_places, left_on='idplace', right_on='id', how='inner')
print(f"Total valid places: {len(df_merged)}")

np.random.seed(42)
shuffled_indices = np.random.permutation(len(df_merged))
split_idx = len(df_merged) // 2

train_df = df_merged.iloc[shuffled_indices[:split_idx]].copy().reset_index(drop=True)
test_df  = df_merged.iloc[shuffled_indices[split_idx:]].copy().reset_index(drop=True)

print(f"Train (queries): {len(train_df)} | Test (search space): {len(test_df)}")

### 2. Type-Aware Evaluation Functions

The key improvement: each place type only uses its **own** relevant subcategory attributes, preventing spurious cross-type matches.

In [ ]:
def eval_level_1(query_typeR, sorted_test_typeR_list):
    if pd.isna(query_typeR) or query_typeR not in sorted_test_typeR_list:
        return None
    for i, t in enumerate(sorted_test_typeR_list):
        if t == query_typeR:
            return i
    return None

def extract_subcategories(row):
    """
    Type-Aware subcategory extraction — avoids spurious cross-type matches.
    Each typeR only draws from its own relevant metadata columns:
      H  (Hotel)          -> priceRange
      R  (Restaurant)     -> restaurantType, restaurantTypeCuisine
      A / AP (Attraction) -> activiteSubType
    """
    cats = set()
    type_r = str(row.get('typeR', '')).strip()
    if type_r == 'H':
        val = row.get('priceRange', '')
        if pd.notna(val) and str(val).strip():
            for p in str(val).split(','):
                cats.add(p.strip().lower())
    elif type_r == 'R':
        for col in ['restaurantType', 'restaurantTypeCuisine']:
            val = row.get(col, '')
            if pd.notna(val) and str(val).strip():
                for p in str(val).split(','):
                    cats.add(p.strip().lower())
    elif type_r in ('A', 'AP'):
        val = row.get('activiteSubType', '')
        if pd.notna(val) and str(val).strip():
            for p in str(val).split(','):
                cats.add(p.strip().lower())
    return cats

test_subcats_list = [extract_subcategories(row) for _, row in test_df.iterrows()]

def eval_level_2(query_subcats, sorted_test_indices):
    if not query_subcats:
        return None
    for i, test_idx in enumerate(sorted_test_indices):
        if query_subcats.intersection(test_subcats_list[test_idx]):
            return i
    return None

print("Type-aware evaluation functions ready.")

### 3. Build BM25 Index + Transformer Vectors

In [ ]:
train_texts = train_df['top_100_words'].fillna('').tolist()
test_texts  = test_df['top_100_words'].fillna('').tolist()

# --- BM25 ---
print("Building BM25 index on test corpus...")
tokenized_corpus = [doc.split() for doc in test_texts]
bm25 = BM25Okapi(tokenized_corpus)
print(f"BM25 indexed {len(tokenized_corpus)} documents.")

# --- Sentence Transformer ---
print("\nEncoding test set with all-MiniLM-L6-v2...")
model = SentenceTransformer('all-MiniLM-L6-v2')
t0 = time.time()
test_vectors  = model.encode(test_texts,  batch_size=64, show_progress_bar=True)
train_vectors = model.encode(train_texts, batch_size=64, show_progress_bar=True)
print(f"Encoding done in {time.time()-t0:.2f}s")

# Pre-compute full cosine similarity matrix
trans_sims = cosine_similarity(train_vectors, test_vectors)
print(f"Similarity matrix shape: {trans_sims.shape}")

### 4. Model F — Optimized Score Fusion (85% BM25 + 15% Transformer)

BM25 scores are min-max normalized to [0,1] per query before fusion, so both signals are on a comparable scale.

The 85/15 weighting was determined empirically to maximize Level 2 improvement while minimizing Level 1 regression relative to pure BM25.

In [ ]:
BM25_WEIGHT   = 0.85
TRANSFO_WEIGHT = 0.15

lvl1_errors = []
lvl2_errors = []
test_type_array = test_df['typeR'].values

start_time = time.time()
print(f"Evaluating {len(train_df)} queries (Model F: {BM25_WEIGHT:.0%} BM25 + {TRANSFO_WEIGHT:.0%} Transformer)...")

for i in range(len(train_df)):
    row = train_df.iloc[i]
    query_text = str(row['top_100_words']).strip()
    if not query_text or query_text == 'nan':
        continue

    # --- BM25 scores ---
    bm25_scores = bm25.get_scores(query_text.split())
    
    # Min-max normalize BM25 to [0, 1]
    bm25_max = np.max(bm25_scores)
    if bm25_max > 0:
        bm25_norm = (bm25_scores - np.min(bm25_scores)) / (bm25_max - np.min(bm25_scores) + 1e-9)
    else:
        bm25_norm = bm25_scores

    # --- Transformer cosine similarities ---
    trans_scores = trans_sims[i]  # already in [-1, 1], predominantly [0, 1]

    # --- Fused ranking ---
    hybrid_scores = BM25_WEIGHT * bm25_norm + TRANSFO_WEIGHT * trans_scores
    ranked_indices = np.argsort(hybrid_scores)[::-1]

    # --- Level 1 ---
    test_typeR_list = test_type_array[ranked_indices].tolist()
    err_1 = eval_level_1(row['typeR'], test_typeR_list)
    if err_1 is not None:
        lvl1_errors.append(err_1)

    # --- Level 2 ---
    query_subcats = extract_subcategories(row)
    err_2 = eval_level_2(query_subcats, ranked_indices)
    if err_2 is not None:
        lvl2_errors.append(err_2)

print(f"Evaluation done in {time.time() - start_time:.2f}s")
print("-" * 60)
print(f"Model F (85% BM25 + 15% Transformer) — Level 1 Error: {np.mean(lvl1_errors):.2f} (on {len(lvl1_errors)} queries)")
print(f"Model F (85% BM25 + 15% Transformer) — Level 2 Error: {np.mean(lvl2_errors):.2f} (on {len(lvl2_errors)} queries)")
print("-" * 60)
print("\n--- Full Comparison Table (Type-Aware Evaluation) ---")
print(f"{'Model':<45} {'L1 Error':>10} {'L2 Error':>10}")
print("-" * 67)
print(f"{'BM25 Baseline':<45} {'0.64':>10} {'4.41':>10}")
print(f"{'Model A (TF-IDF + Cosine)':<45} {'0.68':>10} {'4.74':>10}")
print(f"{'Model B (MiniLM word bag)':<45} {'0.79':>10} {'4.84':>10}")
print(f"{'Model F (85% BM25 + 15% Transformer)':<45} {np.mean(lvl1_errors):.2f} {np.mean(lvl2_errors):.2f}")